# HS4002 Week 5

## Basics of OLS Regression

Today we'll learn linear regression through one research question: **to what extent does job prestige predict income?**

We'll build three models:
- **Model 1**: income ~ prestige
- **Model 2**: income ~ prestige + gender
- **Model 3**: income ~ prestige + gender + age

In Python, `statsmodels` provides OLS regression with a **formula API** that mirrors R's `lm()` syntax almost exactly.

In [4]:
# !pip install pandas numpy plotnine statsmodels stargazer

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

## Data Setup

In [5]:
raw_df = pd.read_csv("gss2022_mini.csv")

chosen_variables = [
	'id', 'age', 'yrlvmus', 'yrartxbt', 'yrmovie', 'yrcreat',
	'income16', 'prestg10', 'degree', 'race', 'sex', 'wrkstat'
]
df = raw_df[chosen_variables].dropna().copy()

# Recode income16 to continuous midpoint values
income_map = {
	1: 500,    2: 2000,   3: 3500,   4: 4500,   5: 5500,   6: 6500,
	7: 7500,   8: 9000,   9: 11250,  10: 13750, 11: 16250, 12: 18750,
	13: 21250, 14: 23750, 15: 27500, 16: 32500, 17: 37500, 18: 45000,
	19: 55000, 20: 67500, 21: 82500, 22: 100000, 23: 120000, 24: 140000,
	25: 160000, 26: 250000
}
df['income_cont'] = df['income16'].map(income_map)

# Binary cultural participation indicators
df['binary_lvmus']  = (df['yrlvmus']  == 1).astype(int)
df['binary_artxbt'] = (df['yrartxbt'] == 1).astype(int)
df['binary_movie']  = (df['yrmovie']  == 1).astype(int)
df['binary_creat']  = (df['yrcreat']  == 1).astype(int)
df['omni'] = df[['binary_lvmus','binary_artxbt','binary_movie','binary_creat']].sum(axis=1)

# Derived variables
df['race_bin']   = df['race'].astype(str)
df['ba_binary']  = (df['degree'] >= 3).astype(int)
df['sex_woman']  = np.where(df['sex'] == 2, 'Woman', 'Not Woman')
df['working']    = np.where(df['wrkstat'] <= 3, 'Working', 'Not Working')

# Model 1

Regress income (`income_cont`) on occupational prestige (`prestg10`).

Hint: `smf.ols('y ~ x', data=df).fit()` is the Python equivalent of `lm(y ~ x, data=df)` in R.

In [6]:
lm1 = smf.ols('income_cont ~ prestg10', data=df).fit()

## Print a summary of `lm1`

Equivalent to R's `summary(lm1)`.

In [7]:
print(lm1.summary())

                            OLS Regression Results                            
Dep. Variable:            income_cont   R-squared:                       0.175
Model:                            OLS   Adj. R-squared:                  0.174
Method:                 Least Squares   F-statistic:                     149.7
Date:                Wed, 09 Sep 2026   Prob (F-statistic):           2.42e-31
Time:                        14:43:33   Log-Likelihood:                -8842.4
No. Observations:                 707   AIC:                         1.769e+04
Df Residuals:                     705   BIC:                         1.770e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -2.001e+04   9120.202     -2.194      0.0

### Provide a substantive interpretation of the results from `lm1`.

In [8]:
# AIC is pretty small and this is not surprising bc few variables
# Focus on the regression coefficients (coef) part, intercept part is beta0
# estimated intercept is -2.001.. and est prestige is 2236
# P > |t| is the comment on the null (which is that coeff occupational prestige is 0)
# we did not get 0, and the p value also shows that this is super incompatible with the null
# 0.000 chance, therefore, reject the null
# 2236 is the beta hat , for every one unit increase, income is estimated to increase by 2236

# Model 2

Add gender (`sex_woman`) to the model. `statsmodels` automatically creates dummy variables for string/categorical columns.

In [9]:
lm2 = smf.ols('income_cont ~ prestg10 + sex_woman', data=df).fit()
print(lm2.summary())

                            OLS Regression Results                            
Dep. Variable:            income_cont   R-squared:                       0.180
Model:                            OLS   Adj. R-squared:                  0.178
Method:                 Least Squares   F-statistic:                     77.45
Date:                Wed, 09 Sep 2026   Prob (F-statistic):           3.96e-31
Time:                        14:43:33   Log-Likelihood:                -8840.2
No. Observations:                 707   AIC:                         1.769e+04
Df Residuals:                     704   BIC:                         1.770e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept          -1.381e+04   9556

In [10]:
# relationship between gender and income Negative 1.044e^4, if you are a woman, you are expected to have a 10k decrease in income, holding occupational prestige constant
# P value is 0.034 is how likely it is that the true coefficient for gender is 0 
#Not very likely and the standard error is 5k 
# If zero, the error can be like 5k, but then coef is 10k, so reject the null
# This is saying holding occupational prestige constant


## Compare Model 1 and Model 2

### Comparing R² and Adjusted R² values

In [11]:
comparison = pd.DataFrame({
	'Model':         ['Model 1', 'Model 2'],
	'R-Squared':     [lm1.rsquared,     lm2.rsquared],
	'Adj. R-Squared':[lm1.rsquared_adj, lm2.rsquared_adj]
})
comparison

,Model,R-Squared,Adj. R-Squared
0,Model 1,0.175109,0.173939
1,Model 2,0.180349,0.178020


In [12]:
# model 2 is better under r square and adj r squared 

### Comparing AIC values

Lower AIC is better.

In [13]:
print(f'AIC Model 1: {lm1.aic:.2f}')
print(f'AIC Model 2: {lm2.aic:.2f}')

AIC Model 1: 17688.88
AIC Model 2: 17686.38


In [14]:
# the number alone is not very meaningful,we need to compare two related models and see the lower one
# lower one is better according to aic , slightly smaller 

### Conducting an F-test

Equivalent to R's `anova(lm1, lm2)`.

This works only for **nested** models — Model 1's predictors must be a subset of Model 2's, fitted on the same rows. It cannot compare `prestige + gender` against `prestige + age`, since neither one contains the other. Remember, the null hypothesis tests if the additional variables in the unrestricted model **all** have coefficients of zero.


In [15]:
# anova_lm compares nested models with an F-test
print(anova_lm(lm1, lm2))

   df_resid           ssr  df_diff       ss_diff         F    Pr(>F)
0     705.0  3.022575e+12      0.0           NaN       NaN       NaN
1     704.0  3.003374e+12      1.0  1.920049e+10  4.500653  0.034231


In [16]:
# assessment for how likely it is that additional variables are gender

### Which model do you prefer? Justify using the statistics above.

In [17]:
# Your answer here

# Model 3

Add age (`age`) to Model 2.

In [18]:
lm3 = smf.ols('income_cont ~ prestg10 + sex_woman + age', data=df).fit()
print(lm3.summary())

                            OLS Regression Results                            
Dep. Variable:            income_cont   R-squared:                       0.181
Model:                            OLS   Adj. R-squared:                  0.178
Method:                 Least Squares   F-statistic:                     51.91
Date:                Wed, 09 Sep 2026   Prob (F-statistic):           2.57e-30
Time:                        14:43:33   Log-Likelihood:                -8839.8
No. Observations:                 707   AIC:                         1.769e+04
Df Residuals:                     703   BIC:                         1.771e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept          -7449.5499   1.17

## Compare Model 2 and Model 3

In [19]:
comparison2 = pd.DataFrame({
	'Model':          ['Model 2', 'Model 3'],
	'R-Squared':      [lm2.rsquared,     lm3.rsquared],
	'Adj. R-Squared': [lm2.rsquared_adj, lm3.rsquared_adj],
	'AIC':            [lm2.aic,          lm3.aic]
})
comparison2

,Model,R-Squared,Adj. R-Squared,AIC
0,Model 2,0.180349,0.178020,17686.375978
1,Model 3,0.181360,0.177866,17687.503692


In [20]:
print(anova_lm(lm2, lm3))

   df_resid           ssr  df_diff       ss_diff         F    Pr(>F)
0     704.0  3.003374e+12      0.0           NaN       NaN       NaN
1     703.0  2.999671e+12      1.0  3.703232e+09  0.867886  0.351861


# Making a professional-looking regression table

The Python `stargazer` package produces LaTeX/HTML tables equivalent to R's `stargazer` or `texreg`.

Run the cell below, then follow the Overleaf steps to turn that output into a typeset PDF table.

## How to use Overleaf

Overleaf is a LaTeX editor that runs in your browser. There is nothing to install, it compiles your document to PDF for you, and teammates can edit the same file at once. You'll be writing the research paper in it, so get comfortable now.

### 1. Set up your project

1. Go to [overleaf.com](https://www.overleaf.com) and register for a free account.
2. Open the course template: [https://www.overleaf.com/read/hqjkhwkfnfbr#d8258f](https://www.overleaf.com/read/hqjkhwkfnfbr#d8258f)
3. That link is **read-only**, so you can't type in it. Click **Menu → Copy Project** to get your own editable copy. (If that fails, click **New Project → Blank Project** and paste the template's text in yourself.)

### 2. Paste the table in

Copy the whole block the cell above printed — everything from `\begin{table}` down to `\end{table}` — and paste it between `\begin{document}` and `\end{document}`. Then click **Recompile**.

You'll have made your first LaTeX table!

In [21]:
%pip install stargazer

Note: you may need to restart the kernel to use updated packages.


In [22]:
from stargazer.stargazer import Stargazer

sg = Stargazer([lm1, lm2, lm3])
sg.title('Regression Results')

# LaTeX reads an underscore as a subscript, so the raw name 'income_cont'
# would stop the document from compiling in Overleaf.
sg.dependent_variable_name('Income (USD)')

sg.custom_columns(['Model 1', 'Model 2', 'Model 3'], [1, 1, 1])
sg.covariate_order(['prestg10', 'sex_woman[T.Woman]', 'age', 'Intercept'])
sg.rename_covariates({
	'prestg10':           'Occupational prestige',
	'sex_woman[T.Woman]': 'Gender (woman = 1)',
	'age':                'Age',
	'Intercept':          'Intercept'
})

# Render as LaTeX — paste into Overleaf
print(sg.render_latex())

\begin{table}[!htbp] \centering
  \caption{Regression Results}
\begin{tabular}{@{\extracolsep{5pt}}lccc}
\\[-1.8ex]\hline
\hline \\[-1.8ex]
& \multicolumn{3}{c}{\textit{Dependent variable: Income (USD)}} \
\cr \cline{2-4}
\\[-1.8ex] & \multicolumn{1}{c}{Model 1} & \multicolumn{1}{c}{Model 2} & \multicolumn{1}{c}{Model 3}  \\
\\[-1.8ex] & (1) & (2) & (3) \\
\hline \\[-1.8ex]
 Occupational prestige & 2236.564$^{***}$ & 2221.175$^{***}$ & 2234.384$^{***}$ \\
& (182.823) & (182.515) & (183.082) \\
 Gender (woman = 1) & & -10442.232$^{**}$ & -10668.916$^{**}$ \\
& & (4922.158) & (4928.630) \\
 Age & & & -132.539$^{}$ \\
& & & (142.270) \\
 Intercept & -20009.515$^{**}$ & -13805.519$^{}$ & -7449.550$^{}$ \\
& (9120.202) & (9556.103) & (11742.412) \\
\hline \\[-1.8ex]
 Observations & 707 & 707 & 707 \\
 $R^2$ & 0.175 & 0.180 & 0.181 \\
 Adjusted $R^2$ & 0.174 & 0.178 & 0.178 \\
 Residual Std. Error & 65477.783 (df=705) & 65315.822 (df=704) & 65321.952 (df=703) \\
 F Statistic & 149.658$^{***}

# Bonus Extension

Build a larger model (`lm_w`) using additional socioeconomic covariates available in the data. Compare its coefficient estimates and model-fit statistics to Models 1–3. Is it a better model? Justify your choice.

In [26]:
df.columns.tolist()

['id',
 'age',
 'yrlvmus',
 'yrartxbt',
 'yrmovie',
 'yrcreat',
 'income16',
 'prestg10',
 'degree',
 'race',
 'sex',
 'wrkstat',
 'income_cont',
 'binary_lvmus',
 'binary_artxbt',
 'binary_movie',
 'binary_creat',
 'omni',
 'race_bin',
 'ba_binary',
 'sex_woman',
 'working']

In [27]:
lm_w = smf.ols('income_cont ~ prestg10 + sex_woman + age + degree', data=df).fit()
print(lm_w.summary())

                            OLS Regression Results                            
Dep. Variable:            income_cont   R-squared:                       0.263
Model:                            OLS   Adj. R-squared:                  0.259
Method:                 Least Squares   F-statistic:                     62.76
Date:                Thu, 10 Sep 2026   Prob (F-statistic):           2.33e-45
Time:                        09:50:57   Log-Likelihood:                -8802.4
No. Observations:                 707   AIC:                         1.761e+04
Df Residuals:                     702   BIC:                         1.764e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept          -5050.1794   1.11

In [29]:
# What this step did was adding educational attainment as a fourth explanatory variable
# Adjusted R^2 is increased to 0.259, signifying a substantial increase in fit, allowing the model to explain variance in income
# degree coefficient is 18,960, with p < 0.001, showing that higher edu has a large, statistically significant positive association with income
# Bias has also decreased from 2,234 in model 3 to 1,267 (for prestige) 
# Gender gap became bigger in this mode, -14,530 in income as compared to -10,067 in income, showing a persistent income gap for gender 


In [30]:
import pandas as pd

comparison_all = pd.DataFrame({
    'Model':          ['Model 1', 'Model 2', 'Model 3', 'Model W'],
    'R-Squared':      [lm1.rsquared,     lm2.rsquared,     lm3.rsquared,     lm_w.rsquared],
    'Adj. R-Squared': [lm1.rsquared_adj, lm2.rsquared_adj, lm3.rsquared_adj, lm_w.rsquared_adj],
    'AIC':            [lm1.aic,          lm2.aic,          lm3.aic,          lm_w.aic]
})
comparison_all

,Model,R-Squared,Adj. R-Squared,AIC
0,Model 1,0.175109,0.173939,17688.881424
1,Model 2,0.180349,0.178020,17686.375978
2,Model 3,0.181360,0.177866,17687.503692
3,Model W,0.263408,0.259211,17614.836657


In [31]:
from statsmodels.stats.anova import anova_lm

# Compare Model 3 against Model W
print(anova_lm(lm3, lm_w))

   df_resid           ssr  df_diff       ss_diff          F        Pr(>F)
0     703.0  2.999671e+12      0.0           NaN        NaN           NaN
1     702.0  2.699028e+12      1.0  3.006435e+11  78.195467  7.476192e-18


In [32]:
# model w is also good because of the drop in AIC, from 17687 in model 3 to 17614 in model w, with lower AICs representing goodness of fit and model simplicity 
# we reject the null hypothesis and conclude that adding degree improves model fit to a statistically significant degree